# Inter-annotator agreement

Cohen's kappa for two comparisons: the labels of the first run against the human annotation of a
sample of conversations, and against the second run of `03_classification_rerun`.

* **Input:** `labeled_conversations/`, `validation/llm_rerun/`, `source/human_annotations/`
* **Output:** `results/tables/table_3_2_agreement.csv`
* **Check:** Human-LLM kappa 0.648 / 0.702 / 0.685, LLM-LLM kappa 0.813 / 0.772 / 0.703


In [11]:
import pandas as pd
from pathlib import Path
import os
import json

# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

CODING_SCHEMES = ['lin_dialogue_acts',
                  'acosta_epistemic_stance',
                  'del_bosque_cma']

SCHEME_NAMES = {'lin_dialogue_acts': 'Dialogue act Scheme',
                'acosta_epistemic_stance': 'Epistemic Stance',
                'del_bosque_cma': 'Conversational Management Acts'}

# Read the three label sets

In [12]:
encodings_original = {}
# Read dataframe
for coding_scheme in CODING_SCHEMES:
    df = pd.read_csv(DATA / 'labeled_conversations' / (coding_scheme + '.csv'))

    encodings_original[coding_scheme] = df

In [13]:
encodings_second_labeling = {}
# Read dataframe
for coding_scheme in CODING_SCHEMES:
    df = pd.read_csv(DATA / 'validation' / 'llm_rerun' / (coding_scheme + '.csv'))

    encodings_second_labeling[coding_scheme] = df

In [14]:
encodings_human = {}
# Read dataframe
for coding_scheme in CODING_SCHEMES:
    df = pd.read_csv(DATA / 'source' / 'human_annotations' / (coding_scheme + '.csv'))
    df = df.rename(columns={'human_label': 'label'})

    encodings_human[coding_scheme] = df

encodings_human['lin_dialogue_acts'].head()

,activity_id,user_id,role,message_num,utterance_num,sentence_en,label
0,asst_Get3WE5ozTjkHcdGhnqGGqgL,s103,model,0.0,0.0,"Hello! 😸 I am SIMBA, and I will help you think...",Greeting
1,asst_Get3WE5ozTjkHcdGhnqGGqgL,s103,model,0.0,1.0,"**Question 1:** In your opinion, what are the ...",Probing Question
2,asst_Get3WE5ozTjkHcdGhnqGGqgL,s103,model,0.0,2.0,**Question 2:** What are the points of the cou...,Probing Question
3,asst_Get3WE5ozTjkHcdGhnqGGqgL,s103,model,0.0,3.0,Your turn! 🌟,Directive
4,asst_Get3WE5ozTjkHcdGhnqGGqgL,s103,user,1.0,0.0,"In this course, we saw the notions of critical...",Information


# Merge each validation set with the labels of the first run

The merge is left on the first run, so that the utterances the validation set does not cover come back
as NaN and are counted as "not compared".

In [15]:
def merge_encodings(encodings_validation):
    merged_encodings = {}
    for coding_scheme in CODING_SCHEMES:
        merged_df = encodings_original[coding_scheme].merge(
            encodings_validation[coding_scheme][['activity_id', 'user_id', 'role', 'message_num',
                                                 'utterance_num', 'label']],
            on=['activity_id', 'user_id', 'role', 'message_num', 'utterance_num'],
            how='left',
            suffixes=('', '_validation')
        )

        merged_encodings[coding_scheme] = merged_df

    return merged_encodings

# Add the first-level category of the validation labels

In [16]:
codings = {}

for coding_scheme in CODING_SCHEMES:
    # Read coding scheme
    path = Path('..') / 'coding_schemes' / coding_scheme
    with open(path / 'tags.json') as f:
        scheme = json.load(f)
        codings[coding_scheme] = scheme


CATEGORY_COLUMN = 'type_validation'
LABEL_COLUMN = 'label_validation'


def add_category(merged_encodings):
    for coding_scheme in CODING_SCHEMES:
        reversed_coding_scheme = {label: category
                                  for category, labels in codings[coding_scheme].items()
                                  for label in labels}
        df = merged_encodings[coding_scheme].copy()
        df[CATEGORY_COLUMN] = df[LABEL_COLUMN].map(reversed_coding_scheme).fillna('Undefined')
        merged_encodings[coding_scheme] = df.copy()

    return merged_encodings

# Cohen's kappa

In [17]:
from sklearn.metrics import cohen_kappa_score


def compute_metrics(merged_encodings, comparison):
    metrics = {}

    for coding_scheme in CODING_SCHEMES:
        df = merged_encodings[coding_scheme].copy()

        # Count rows with NaN in each column separately
        nan_in_label_validation = df['label_validation'].isna().sum()
        total_labels = len(list(df['label']))
        compared_rows = total_labels - nan_in_label_validation
        # Percentage compared
        percentage = (total_labels - nan_in_label_validation) * 100 / total_labels

        df = df.dropna(subset=['label', 'label_validation'])

        # Compute Cohen's kappa for the second-level label
        kappa_label = cohen_kappa_score(list(df['label']), list(df['label_validation']))

        # Compute Cohen's kappa for the first-level label
        kappa_type = cohen_kappa_score(list(df['type']), list(df['type_validation']))

        metric = {'coding_scheme': SCHEME_NAMES[coding_scheme],
                  'comparison': comparison,
                  'utterances': compared_rows,
                  'pct_of_total': round(percentage, 2),
                  'kappa_first_level': round(kappa_type, 3),
                  'kappa_second_level': round(kappa_label, 3)}
        metrics[coding_scheme] = metric

        print(coding_scheme.upper())
        for key in metric:
            print(key, ':', metric[key])
        print('')

    return metrics


metrics_human = compute_metrics(add_category(merge_encodings(encodings_human)), 'Human-LLM')

LIN_DIALOGUE_ACTS
coding_scheme : Dialogue act Scheme
comparison : Human-LLM
utterances : 353
pct_of_total : 7.24
kappa_first_level : 0.648
kappa_second_level : 0.385

ACOSTA_EPISTEMIC_STANCE
coding_scheme : Epistemic Stance
comparison : Human-LLM
utterances : 314
pct_of_total : 6.37
kappa_first_level : 0.702
kappa_second_level : 0.702

DEL_BOSQUE_CMA
coding_scheme : Conversational Management Acts
comparison : Human-LLM
utterances : 343
pct_of_total : 6.91
kappa_first_level : 0.685
kappa_second_level : 0.685



In [18]:
metrics_llm = compute_metrics(add_category(merge_encodings(encodings_second_labeling)), 'LLM-LLM')

LIN_DIALOGUE_ACTS
coding_scheme : Dialogue act Scheme
comparison : LLM-LLM
utterances : 4658
pct_of_total : 95.51
kappa_first_level : 0.813
kappa_second_level : 0.728

ACOSTA_EPISTEMIC_STANCE
coding_scheme : Epistemic Stance
comparison : LLM-LLM
utterances : 4882
pct_of_total : 99.07
kappa_first_level : 0.772
kappa_second_level : 0.772

DEL_BOSQUE_CMA
coding_scheme : Conversational Management Acts
comparison : LLM-LLM
utterances : 4921
pct_of_total : 99.15
kappa_first_level : 0.703
kappa_second_level : 0.703



# Table 3.2

In [19]:
table_3_2 = pd.DataFrame([metrics_human[coding_scheme] for coding_scheme in CODING_SCHEMES]
                         + [metrics_llm[coding_scheme] for coding_scheme in CODING_SCHEMES])
table_3_2 = table_3_2.sort_values(['coding_scheme', 'comparison'])

path_out = DATA / 'results' / 'tables'
path_out.mkdir(parents=True, exist_ok=True)
table_3_2.to_csv(path_out / 'table_3_2_agreement.csv', index=False)

table_3_2

,coding_scheme,comparison,utterances,pct_of_total,kappa_first_level,kappa_second_level
2,Conversational Management Acts,Human-LLM,343,6.91,0.685,0.685
5,Conversational Management Acts,LLM-LLM,4921,99.15,0.703,0.703
0,Dialogue act Scheme,Human-LLM,353,7.24,0.648,0.385
3,Dialogue act Scheme,LLM-LLM,4658,95.51,0.813,0.728
1,Epistemic Stance,Human-LLM,314,6.37,0.702,0.702
4,Epistemic Stance,LLM-LLM,4882,99.07,0.772,0.772
